# 1. `omnipath-client` tour

`omnipath-client` is a lightweight HTTP client for the new OmniPath API
(`dev.omnipathdb.org`). The pilot build serves 15 resources; many are
small-molecule / metabolomics related, including the food-compound
databases.

Status: **pre-alpha**, expect rough edges and breaking changes.

In [ ]:
from _utils import results_dir
import omnipath_client as oc

## What endpoints are exposed?

In [ ]:
endpoints = oc.endpoints()
list(endpoints)[:10]

## Discover the parameters of one endpoint

In [ ]:
oc.params("interactions")

In [ ]:
oc.values("interactions", "resources")[:20]

## Fetch a small slice of interactions

Default return is a polars DataFrame; you can ask for pandas or pyarrow
via the client constructor.

In [ ]:
df = oc.interactions(
    resources=["SignaLink3", "SIGNOR"],
    organisms=[9606],
)
df.head()

In [ ]:
print(f"{df.height} rows × {df.width} columns")
df.columns

## Pandas backend

Construct a client explicitly when you want a different backend.

In [ ]:
client = oc.OmniPath(backend="pandas")
df_pd = client.interactions(resources=["SignaLink3"], organisms=[9606])
type(df_pd), df_pd.shape

## Search the ontology terms

In [ ]:
oc.search_terms(["glycolysis", "tca cycle"], limit=5)

**Recap.** `oc.endpoints / params / values` introspect what's available;
`oc.interactions / entities / associations` fetch data; ontology helpers
look up biological terms. In script 02 we'll use the utils API for ID
translation.

# 2. ID translation & orthology — the utils API

`omnipath_client.utils` is a thin wrapper around the
`utils.omnipathdb.org` web service. It serves the same translation /
taxonomy / orthology logic that the legacy R `OmnipathR::translate_ids()`
wraps locally — but as an HTTP service, so you don't need to download
the resource files.

This is the Python parallel of the R `03_id_translation.R` section.

In [ ]:
from _utils import results_dir
from omnipath_client import utils

## Available ID types

In [ ]:
utils.id_types("uniprot")[:10]

## Translate one identifier

In [ ]:
utils.map_name("TP53", "genesymbol", "uniprot")

## Translate many at once

`map_names()` returns the union; `translate_column()` keeps the
row-to-row mapping as a DataFrame.

In [ ]:
ids = ["TP53", "MYC", "CDK2", "BRCA1"]
utils.map_names(ids, "genesymbol", "uniprot")

In [ ]:
df = utils.translation_df(ids, "genesymbol", "uniprot")
df

## Metabolite ID translation

The same mechanism handles small-molecule IDs.

In [ ]:
utils.map_name("HMDB0000094", "hmdb", "chebi")  # citrate

In [ ]:
metabolites = ["HMDB0000094", "HMDB0000254", "HMDB0000190"]  # citrate, succinate, lactate
utils.translation_df(metabolites, "hmdb", "chebi")

## Taxonomy resolution

Names, IDs, codes — `resolve_organism` handles them all.

In [ ]:
utils.ensure_ncbi_tax_id("human"), utils.ensure_ncbi_tax_id("Mus musculus"), utils.ensure_ncbi_tax_id(10090)

## Orthology

Translate a list of human gene symbols to mouse.

In [ ]:
from omnipath_client.utils._orthology import orthology_df

orthology_df(
    source=9606,
    target=10090,
    id_type="genesymbol",
    identifiers=["TP53", "MYC", "CDK2"],
)

**Recap.** ID translation, taxonomy resolution, and orthology — all via
a small handful of utility calls. This is the bridge that lets us
combine human-only resources with multi-organism analyses, which we'll
use when fetching COSMOS PKN data in script 03.

# 3. COSMOS PKN — fetched via the client

**COSMOS** is a multi-omics mechanistic-modelling framework that
combines metabolomics, signalling, and transcriptomics. Its Prior
Knowledge Network (PKN) is a multi-layer graph stitched together from:

- **transporters** — TCDB, SLC, GEM, Recon3D, MRCLinksDB
- **receptors** — MRCLinksDB, STITCH
- **allosteric regulation** — BRENDA, STITCH
- **enzyme ↔ metabolite** — Human-GEM, Recon3D
- **PPI** — OmniPath
- **GRN** — CollecTRI

We won't run COSMOS itself today, but we'll inspect the PKN and the
GEM-derived layer because the binary network (reaction → reactant /
product) is generally useful for any metabolic-network reasoning.

In [ ]:
from _utils import results_dir
from omnipath_client import cosmos

## What's available?

In [ ]:
cosmos.categories()

In [ ]:
cosmos.organisms()

In [ ]:
cosmos.resources(organism="human")

## Status of the live build

In [ ]:
cosmos.status()

## Fetch the enzyme ↔ metabolite layer

This is the layer that captures "enzyme E catalyses a reaction
producing/consuming metabolite M". The server delivers it pre-cleaned
(canonical ChEBI / UniProt IDs, metabolic vs transport split).

In [ ]:
em_df = cosmos.get_pkn(
    organism="human",
    categories="enzyme_metabolite",
    format="dataframe",
)
em_df.head()

In [ ]:
print(f"{em_df.height} rows × {em_df.width} columns")
em_df.columns

## A small subnetwork for visualisation

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Take a small slice to keep the plot readable.
import polars as pl

slice_df = em_df.head(80).select(["source", "target", "interaction_type"])
G = nx.from_pandas_edgelist(
    slice_df.to_pandas(),
    source="source",
    target="target",
    edge_attr="interaction_type",
    create_using=nx.DiGraph,
)

fig, ax = plt.subplots(figsize=(9, 7))
pos = nx.spring_layout(G, seed=42, k=0.7)
nx.draw_networkx_nodes(G, pos, node_size=80, node_color="#88aadd", ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.4, arrows=True, arrowsize=8, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=6, ax=ax)
ax.set_title("First 80 enzyme ↔ metabolite edges of the COSMOS PKN")
ax.set_axis_off()
fig.savefig(results_dir("03_cosmos") / "cosmos_em_subnet.png", dpi=150, bbox_inches="tight")
plt.show()

## The other categories at a glance

Each is one HTTP call away; the API is uniform.

In [ ]:
for cat in ["transporters", "receptors", "allosteric"]:
    df = cosmos.get_pkn(organism="human", categories=cat)
    print(f"{cat:<14s}  {df.height:>6d} edges")

**Recap.** Three lines of Python and we have a multi-layer
metabolism-aware network. Script 04 shows what the GEM looks like
*before* COSMOS simplifies it.

# 4. Raw GEM reactions via `omnipath-metabo`

`omnipath-metabo` is the server-side builder behind the COSMOS PKN.
Hitting it directly (rather than the client) lets us see resources in
their original shape — before COSMOS simplifies them into a binary
enzyme ↔ metabolite graph.

We focus on **GEM (Genome-scale Metabolic Model) reactions** — these
carry stoichiometry, compartment information, and reaction reversibility,
which the COSMOS-formatted version largely flattens away.

In [ ]:
from _utils import results_dir
from omnipath_metabo.datasets.cosmos.resources import gem_interactions

## A few raw interactions

`gem_interactions()` is a generator that yields one `Interaction`
named-tuple per edge.

In [ ]:
gem_iter = gem_interactions(gem="Human-GEM", organism=9606)
sample = [next(gem_iter) for _ in range(5)]
for rec in sample:
    print(rec)

Each record carries:

- `source` / `target` — IDs (ChEBI for metabolites, UniProt for proteins)
- `source_type` / `target_type` — `'small_molecule'` / `'protein'`
- `interaction_type` — what the edge represents (e.g. `'catalyses'`,
  `'transports'`)
- `resource` — `'GEM:Human-GEM'` for metabolic reactions,
  `'GEM_transporter:Human-GEM'` for transport reactions
- `locations` — compartment(s) where the reaction occurs
- `mor` — mode-of-regulation (sign, where applicable)
- `attrs` — extra resource-specific fields

## Collect everything into a DataFrame

Be aware: this materialises tens of thousands of edges; takes a few
seconds.

In [ ]:
import pandas as pd

records = list(gem_interactions(gem="Human-GEM", organism=9606))
print(f"Total Human-GEM interactions: {len(records)}")

df = pd.DataFrame(
    {
        "source": [r.source for r in records],
        "target": [r.target for r in records],
        "interaction_type": [r.interaction_type for r in records],
        "resource": [r.resource for r in records],
        "locations": [r.locations for r in records],
        "mor": [r.mor for r in records],
    }
)
df.head()

## Metabolic vs transport split

In [ ]:
df["resource"].value_counts()

## Compartment distribution

GEM reactions are localised to a compartment (cytosol, mitochondrion,
extracellular …). Transport reactions cross compartments.

In [ ]:
from collections import Counter

loc_counter: Counter[str] = Counter()
for locs in df["locations"]:
    if locs:
        for loc in locs:
            loc_counter[loc] += 1

pd.Series(loc_counter).sort_values(ascending=False).head(15)

## Slice: a single metabolite's neighbourhood

Pick citrate (`CHEBI:30769`) and look at every reaction it touches.

In [ ]:
citrate = "CHEBI:30769"
near_citrate = df[(df["source"] == citrate) | (df["target"] == citrate)]
print(f"{len(near_citrate)} reactions involve citrate")
near_citrate.head(10)

In [ ]:
df.to_parquet(results_dir("04_gem") / "human_gem_interactions.parquet")

**Recap.** Same data the client serves under `enzyme_metabolite`, but
with stoichiometry and compartment information intact. Useful when you
need to reason about reaction direction, mass balance, or
compartmentalisation rather than just connectivity.

# 5. (OPTIONAL) BRENDA allosteric regulation

**Skip if running short on time.** This section showcases a single
resource — BRENDA — whose depth and curation quality make it worth a
brief stop.

BRENDA catalogues enzyme function with manual literature curation: each
regulator is annotated with the reference(s) and, often, with the
specific kinetic effect. For metabolomics modelling, this is one of the
few places you can find systematic small-molecule → enzyme allosteric
regulation data.

In [ ]:
from _utils import results_dir
from omnipath_metabo.datasets.cosmos.resources import brenda_regulations

## A few raw regulations

In [ ]:
sample = []
gen = brenda_regulations(organism=9606)
for _ in range(5):
    sample.append(next(gen))

for rec in sample:
    print(rec)

## Collect everything

In [ ]:
import pandas as pd

records = list(brenda_regulations(organism=9606))
print(f"Total human BRENDA allosteric regulations: {len(records)}")

df = pd.DataFrame(
    {
        "regulator": [r.source for r in records],
        "enzyme": [r.target for r in records],
        "interaction_type": [r.interaction_type for r in records],
        "mor": [r.mor for r in records],
        "resource": [r.resource for r in records],
    }
)
df.head()

## Mode of regulation distribution

`mor` (mode-of-regulation) carries the sign: activation, inhibition, or
unknown.

In [ ]:
df["mor"].value_counts(dropna=False)

## Most-regulated enzymes

In [ ]:
df.groupby("enzyme").size().sort_values(ascending=False).head(15)

## Most "active" regulators

In [ ]:
df.groupby("regulator").size().sort_values(ascending=False).head(15)

In [ ]:
df.to_parquet(results_dir("05_brenda") / "brenda_allosteric.parquet")

**Wrap up.**

Today we walked from raw metabolomics measurements through differential
analysis, ID translation, prior-knowledge access, enrichment, and into
multi-layer mechanistic networks. Two takeaways:

1. The R side (OmnipathR + MetaProViz) is now a coherent SE-centric
   pipeline; the API is stable, vignettes are up to date, and a
   Bioconductor release lands shortly.
2. The Python side (omnipath-client + omnipath-metabo) is fresh and
   pre-alpha. Bug reports welcome at
   [github.com/saezlab](https://github.com/saezlab).